# Chapter 8 — Coding Exercise
## Build Your Own Estimator & a Full Workflow

Based on **Chapter 8** of *Introduction to Machine Learning with Python*.

## What you'll practice

- Implement a **custom transformer** compatible with scikit-learn
- Drop it into a **`Pipeline`**
- Run a full **grid-searched** workflow
- Compare against a **`DummyClassifier`** baseline

**How to use this notebook:** fill in each cell marked `# TODO`, then run the **Check** cell below it. Full solutions are at the end — try each task yourself first!

> Requires `numpy`, `scikit-learn` (and `pandas` for some chapters).

## Setup

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

## Exercise 1 — A custom scaler

Complete `MyStandardScaler` so that `fit` stores each feature's mean and std, and `transform` returns `(X - mean) / std`. It must inherit from `BaseEstimator` and `TransformerMixin`.

In [ ]:
class MyStandardScaler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # TODO: store self.mean_ and self.std_
        return self
    def transform(self, X):
        # TODO: return the standardized X
        return X

In [ ]:
# Check
s = MyStandardScaler().fit(X_train)
Xs = s.transform(X_train)
assert hasattr(s, "mean_") and hasattr(s, "std_"), "Store mean_ and std_ in fit()."
assert np.allclose(Xs.mean(axis=0), 0, atol=1e-6), "Columns should have mean ~0."
print("Custom scaler works — column means ~0, stds ~1")

## Exercise 2 — Use it in a pipeline

Put `MyStandardScaler` and `LogisticRegression(max_iter=5000)` in a `Pipeline` and compute 5-fold `cross_val_score` on the training data.

In [ ]:
# TODO: build `pipe`, compute `cv_scores`
pipe = None
cv_scores = None

## Exercise 3 — A full grid-searched workflow

Build a `Pipeline` of `MyStandardScaler` + `SVC`, grid-search `svc__C` and `svc__gamma` in `[0.01, 0.1, 1, 10, 100]`, and print the best params and **test** score.

In [ ]:
# TODO: build and fit `grid`
grid = None

In [ ]:
# Check
assert grid is not None
acc = grid.score(X_test, y_test)
print("best:", grid.best_params_, "| test:", round(acc, 3))
assert acc > 0.9

## Exercise 4 — Baseline comparison

Fit a `DummyClassifier(strategy="most_frequent")` and compare its test accuracy to your grid-searched model. Did the model beat the baseline?

In [ ]:
# TODO: fit the dummy and print both accuracies
# YOUR CODE HERE

---
## Solutions

In [ ]:
class MyStandardScaler(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        return self
    def transform(self, X):
        return (X - self.mean_) / self.std_

# Ex2
pipe = Pipeline([("scaler", MyStandardScaler()),
                 ("logreg", LogisticRegression(max_iter=5000))])
cv_scores = cross_val_score(pipe, X_train, y_train, cv=5)
print("Ex2 mean CV:", round(cv_scores.mean(), 3))

# Ex3
pipe2 = Pipeline([("scaler", MyStandardScaler()), ("svc", SVC())])
param_grid = {"svc__C": [0.01, 0.1, 1, 10, 100],
              "svc__gamma": [0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(pipe2, param_grid, cv=5).fit(X_train, y_train)
print("Ex3 best:", grid.best_params_, "test:", round(grid.score(X_test, y_test), 3))

# Ex4
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Ex4 dummy test :", round(dummy.score(X_test, y_test), 3))
print("Ex4 model test :", round(grid.score(X_test, y_test), 3))